<center><h1>Groome_Henry_HW3</h1></center>
<br>
<br>

Name: Henry Groome
<br>
Github Username: hgroome46
<br>
USC ID: 4479346040

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
data_dir = 'AReM'
activities = sorted(d for d in os.listdir(data_dir)
                    if os.path.isdir(os.path.join(data_dir, d)))
print(activities)

instances = []   
for activity in activities:
    folder = os.path.join(data_dir, activity)
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith('.csv'):
            continue
        num = int(fname.replace('dataset', '').replace('.csv', ''))
        df = pd.read_csv(os.path.join(folder, fname), skiprows=5, names=cols,
                 usecols=range(7), sep=r'[,\s]+', engine='python')
        instances.append({'activity': activity, 'num': num, 'data': df})

print(len(instances), 'files loaded')


['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']


NameError: name 'cols' is not defined

### (b) Test and Train Data

In [ ]:
def is_test(activity, num):
    if activity in ('bending1', 'bending2'):
        return num <= 2
    return num <= 3

for inst in instances:
    inst['split'] = 'test' if is_test(inst['activity'], inst['num']) else 'train'

### (c) Feature Extraction

#### i. Research

Time-domain features commonly used in time series classification:
- Central tendency: mean, median
- Spread: standard deviation, variance, range, interquartile range
- Extremes and position: minimum, maximum, quartiles/percentiles
- Shape of the distribution: skewness, how heavy the tails are
- Energy / magnitude: root mean square (RMS), sum of squares
- Change over time: mean absolute difference between consecutive values, number of zero/mean - crossings, slope of a fitted trend line, autocorrelation

#### ii. Extraction

In [ ]:
series = cols[1:]  

rows = []
for inst in instances:
    df = inst['data']
    row = {'activity': inst['activity'], 'num': inst['num'], 'split': inst['split']}
    for i, s in enumerate(series, start=1):
        x = df[s]
        row[f'min{i}']       = x.min()
        row[f'max{i}']       = x.max()
        row[f'mean{i}']      = x.mean()
        row[f'median{i}']    = x.median()
        row[f'std{i}']       = x.std()
        row[f'1st_quart{i}'] = x.quantile(0.25)
        row[f'3rd_quart{i}'] = x.quantile(0.75)
    rows.append(row)

features = pd.DataFrame(rows)
features.index = range(1, len(features) + 1)  
features.head()

,activity,num,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [ ]:
feature_cols = features.columns[3:]         
feature_std = features[feature_cols].std()
feature_std

rng = np.random.default_rng(42)
n_boot = 1000

results = []
for col in feature_cols:
    x = features[col].values

    boot_stds = []
    for _ in range(n_boot):
        sample = rng.choice(x, size=len(x), replace=True)
        boot_stds.append(np.std(sample, ddof=1))

    lower, upper = np.percentile(boot_stds, [5, 95])
    results.append({'feature': col,
                    'std': np.std(x, ddof=1),
                    'CI_lower (5%)': lower,
                    'CI_upper (95%)': upper})

std_ci = pd.DataFrame(results).set_index('feature')
std_ci

,std,CI_lower (5%),CI_upper (95%)
feature,,,
min1,9.569975,8.194325,10.848390
max1,4.394362,3.328131,5.267411
mean1,5.335718,4.666422,5.871471
median1,5.440054,4.800392,5.977818
std1,1.772153,1.562906,1.948855
1st_quart1,6.153590,5.525727,6.665116
3rd_quart1,5.138925,4.341983,5.876490
min2,0.000000,0.000000,0.000000
max2,5.062729,4.587641,5.411003


#### iv. Select Features

1. mean: the typical signal level, which reflects posture and sensor distance (and is nearly identical to median)
2. std: how much the signal fluctuates, which reflects how much the person is moving. None of the other six statistics measure spread directly.
3. max: the extreme peaks, which captures sudden movements that the mean smooths over.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression line contains the linear regression model within, but has added flexibility. Because of this, there is no way for it's training RSS to be higher than the linear regression model, since more flexibility means a closer fitting line, even if the relationship is ultimately linear. 

### (b) Linear Test

When it comes to test data, the story is different. The cubic model fits the line more closely to noise, which in the training data leads to a lower RSS, but with new test data that we know has a linear relationship, will lead to an overfit line with a higher RSS. So for this case, the linear model has lower RSS.

### (c) Not Linear Train

Again, like in a, the cubic model contains the linear one and so will at least be as closely fitting to the data, but most likely more. If the relationship is nonlinear, then the more flexible cubic model will have a lower RSS with the training data. 

### (d) Not Linear Testing

For test data, we don't have enough information to tell. Because in b, the linear model had a lower RSS because of the data's linearity, it's possible for the linear model to still have a lower RSS than the cubic one if the data has a relationship that is close to linear. The test data could also have a more nonlinear relationship, which would make the cubic model a better fit. So we can't say for sure. 

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 